# Run RMBG 2.0 with Authentication on Modal

This notebook deploys the RMBG-2.0 background removal model and queries its secure web endpoint with proxy authentication.

### Step 1: Deploy the Model

In [ ]:
!uv run modal deploy ../llm-hosting/rmbg-2-0.py

### Step 2: Load Credentials and Setup Endpoint

In [1]:
import os
import requests
import base64

def load_dotenv():
    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        v = v.strip().strip("'\"")
                        os.environ.setdefault(k.strip(), v)
            break

load_dotenv()
MODAL_KEY = os.environ.get("MODAL_KEY")
MODAL_SECRET = os.environ.get("MODAL_SECRET")

if not MODAL_KEY or not MODAL_SECRET:
    print("WARNING: Credentials not loaded!")
else:
    print("Credentials loaded successfully!")

username = "sshibinthomass"
ENDPOINT_URL = f"https://{username}--rmbg-2-0-rmbgmodel-generate-api.modal.run"
EXTRACT_ENDPOINT_URL = f"https://{username}--rmbg-2-0-rmbgmodel-extract-and-remove-ba-111a3b.modal.run"
print(f"Generate Endpoint URL: {ENDPOINT_URL}")
print(f"Extract Endpoint URL: {EXTRACT_ENDPOINT_URL}")

Credentials loaded successfully!
Generate Endpoint URL: https://sshibinthomass--rmbg-2-0-rmbgmodel-generate-api.modal.run
Extract Endpoint URL: https://sshibinthomass--rmbg-2-0-rmbgmodel-extract-and-remove-ba-111a3b.modal.run


### Step 3: Run Authenticated Inference

In [2]:
def test_inference(image_path: str):
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")
        
    headers = {
        "Content-Type": "application/json",
        "Modal-Key": MODAL_KEY,
        "Modal-Secret": MODAL_SECRET,
    }
    payload = {
        "image_base64": image_base64,
    }
    
    print("Sending request to RMBG 2.0 secure endpoint...")
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.content
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

from pathlib import Path
from datetime import datetime

image_path = "../llm-inference/img5.png"
output_bytes = test_inference(image_path)

if output_bytes:
    os.makedirs("outputs", exist_ok=True)
    
    # Format output filename with input file name and current datetime
    input_stem = Path(image_path).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"outputs/{input_stem}_rmbg_{timestamp}.png"
    
    with open(output_filename, "wb") as f:
        f.write(output_bytes)
    print(f"Success! Generated transparent image saved to {output_filename} ({len(output_bytes)} bytes)")

Sending request to RMBG 2.0 secure endpoint...
Success! Generated transparent image saved to outputs/img5_rmbg_20260628_153201.png (1149154 bytes)


In [2]:
def test_extract_inference(image_path: str, target_object: str = None):
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")
        
    headers = {
        "Content-Type": "application/json",
        "Modal-Key": MODAL_KEY,
        "Modal-Secret": MODAL_SECRET,
    }
    payload = {
        "image_base64": image_base64,
        "target_object": target_object,
    }
    
    print("Sending request to RMBG 2.0 secure extract endpoint...")
    response = requests.post(EXTRACT_ENDPOINT_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.content
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

image_path = "../llm-inference/img7.png"
object="laptop"
output_bytes = test_extract_inference(image_path, target_object=object)

if output_bytes:
    os.makedirs("outputs", exist_ok=True)
    from pathlib import Path
    from datetime import datetime
    input_stem = Path(image_path).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"outputs/{input_stem}_extracted_rmbg_{timestamp}.png"
    
    with open(output_filename, "wb") as f:
        f.write(output_bytes)
    print(f"Success! Generated transparent image saved to {output_filename} ({len(output_bytes)} bytes)")

Sending request to RMBG 2.0 secure extract endpoint...
Success! Generated transparent image saved to outputs/img7_extracted_rmbg_20260628_230230.png (1801701 bytes)


### Step 4: Verify Security (Unauthorized Calls)

In [ ]:
print("--- Test: Querying WITHOUT credentials ---")
response = requests.post(ENDPOINT_URL, json={"image_base64": ""})
print(f"Status: {response.status_code} (Expected: 401)")
print(f"Message: {response.text}")